In [10]:
import arcpy
import numpy as np
import random
import math
import os

# 设置工作空间
arcpy.env.workspace = r"D:\ArcGIS\sunleigangPro\模拟退火"
arcpy.env.overwriteOutput = True

# 输入栅格文件
input_raster = "最终ok.tif"

# 输出点要素类
output_points = "output_points.shp"

# 点的高度
point_height = 500  # 单位：米

# 视域范围
view_distance = 150  # 单位：米

# 目标可见区域比例
target_coverage = 0.9  # 85%

# 模拟退火算法参数
initial_temperature = 1000
cooling_rate = 0.99
iterations = 2  # 每次模拟退火的迭代次数

# 将栅格数据加载到 NumPy 数组中
def raster_to_array(raster):
    raster_array = arcpy.RasterToNumPyArray(raster, nodata_to_value=0)
    return raster_array

# 计算栅格的有效面积
def calculate_raster_area(raster):
    cell_size = float(arcpy.GetRasterProperties_management(raster, "CELLSIZEX").getOutput(0))
    raster_array = raster_to_array(raster)
    valid_cell_count = np.count_nonzero(raster_array)
    raster_area = valid_cell_count * (cell_size ** 2)
    return raster_area

# 获取栅格的有效区域多边形
def get_raster_domain(raster):
    # 将栅格的有效区域转换为多边形
    domain_polygon = os.path.join(arcpy.env.workspace, "raster_domain.shp")
    arcpy.RasterDomain_3d(raster, domain_polygon, "POLYGON")
    return domain_polygon

# 计算初始点的数量
def calculate_initial_points(raster_area, view_distance, target_coverage):
    # 每个点的视域范围面积（圆形区域）
    view_area = math.pi * ((view_distance / 2) ** 2)  # 视域范围为圆形，直径为 view_distance
    # 初始点数量 = 栅格面积 * 可见比例 / 视域范围面积
    initial_points = int(raster_area * target_coverage / view_area)
    return initial_points


# Poisson Disk 采样算法
def poisson_disk_sampling(width, height, radius, k=30):
    # 初始化网格和点列表
    cell_size = radius / np.sqrt(2)
    grid_width = int(np.ceil(width / cell_size))
    grid_height = int(np.ceil(height / cell_size))
    grid = [[None for _ in range(grid_width)] for _ in range(grid_height)]
    points = []
    active = []
    
    # 添加初始点
    initial_point = (random.uniform(0, width), random.uniform(0, height))
    points.append(initial_point)
    active.append(initial_point)
    grid[int(initial_point[1] / cell_size)][int(initial_point[0] / cell_size)] = initial_point
    
    # 生成点
    while active:
        random_index = random.randint(0, len(active) - 1)
        point = active[random_index]
        found = False
        
        for _ in range(k):
            angle = random.uniform(0, 2 * np.pi)
            distance = random.uniform(radius, 2 * radius)
            new_point = (point[0] + distance * np.cos(angle), point[1] + distance * np.sin(angle))
            
            if 0 <= new_point[0] < width and 0 <= new_point[1] < height:
                grid_x = int(new_point[0] / cell_size)
                grid_y = int(new_point[1] / cell_size)
                valid = True
                
                for i in range(max(0, grid_x - 2), min(grid_width, grid_x + 3)):
                    for j in range(max(0, grid_y - 2), min(grid_height, grid_y + 3)):
                        neighbor = grid[j][i]
                        if neighbor and np.hypot(neighbor[0] - new_point[0], neighbor[1] - new_point[1]) < radius:
                            valid = False
                            break
                    if not valid:
                        break
                
                if valid:
                    points.append(new_point)
                    active.append(new_point)
                    grid[grid_y][grid_x] = new_point
                    found = True
        
        if not found:
            active.pop(random_index)
    
    return points

# 生成随机点（确保点在栅格的有效区域内，且均匀分布）
def generate_random_points(num_points, domain_polygon):
    # 获取有效区域的范围
    extent = arcpy.Describe(domain_polygon).extent
    width = extent.XMax - extent.XMin
    height = extent.YMax - extent.YMin
    
    # 计算最小距离
    area = width * height
    min_distance = math.sqrt(area / num_points)
    
    # 使用 Poisson Disk 采样生成均匀点
    points = []
    while len(points) < num_points:
        new_points = poisson_disk_sampling(width, height, min_distance)
        
        # 将点转换为实际坐标
        actual_points = [(x + extent.XMin, y + extent.YMin) for x, y in new_points]
        
        # 筛选出位于有效区域内的点
        with arcpy.da.SearchCursor(domain_polygon, ["SHAPE@"]) as cursor:
            for row in cursor:
                polygon = row[0]
                for point in actual_points:
                    if polygon.contains(arcpy.PointGeometry(arcpy.Point(point[0], point[1]))):
                        points.append(point)
        
        # 如果生成的点足够，退出循环
        if len(points) >= num_points:
            break
    
    return points[:num_points]  # 返回指定数量的点

# 计算视域覆盖率
def calculate_coverage(points):
    # 获取输入栅格的坐标系
    spatial_ref = arcpy.Describe(input_raster).spatialReference
    
    # 删除旧的 temp_points.shp 文件
    temp_points = os.path.join(arcpy.env.workspace, "temp_points.shp")
    if arcpy.Exists(temp_points):
        arcpy.management.Delete(temp_points)
    
    # 创建点要素类时设置坐标系
    arcpy.CreateFeatureclass_management(
        arcpy.env.workspace,
        "temp_points.shp",
        "POINT",
        spatial_reference=spatial_ref  # 设置坐标系
    )
    
    # 插入点数据
    with arcpy.da.InsertCursor("temp_points.shp", ["SHAPE@XY"]) as cursor:
        for point in points:
            if not math.isnan(point[0]) and not math.isnan(point[1]):  # 检查坐标是否有效
                cursor.insertRow([point])
    
    # 执行视域分析
    viewshed_result = arcpy.sa.Viewshed2(
        in_raster=input_raster,
        in_observer_features="temp_points.shp",
        out_agl_raster=None,
        analysis_type="FREQUENCY",
        vertical_error="0 Meters",
        out_observer_region_relationship_table=None,
        refractivity_coefficient=0.13,
        surface_offset="0 Meters",
        observer_elevation=point_height,
        observer_offset="1 Meters",
        inner_radius=None,
        inner_radius_is_3d="GROUND",
        outer_radius=view_distance/2,
        outer_radius_is_3d="GROUND",
        horizontal_start_angle=0,
        horizontal_end_angle=360,
        vertical_upper_angle=90,
        vertical_lower_angle=-90,
        analysis_method="ALL_SIGHTLINES",
        analysis_target_device="GPU_THEN_CPU"
    )
    
    # 将结果转换为 NumPy 数组
    viewshed_array = raster_to_array(viewshed_result)
    
    # 计算可见区域面积
    unique_values, counts = np.unique(viewshed_array, return_counts=True)
    cell_size = float(arcpy.GetRasterProperties_management(input_raster, "CELLSIZEX").getOutput(0))
    visible_area = 0
    for value, count in zip(unique_values, counts):
        if value > 0:  # 视域值大于0表示可见区域
            visible_area += count * (cell_size ** 2)
    
    # 计算覆盖率
    coverage = visible_area / raster_area
    print(f"当前覆盖率为: {coverage}")
    return coverage

# 模拟退火算法
def simulated_annealing(points):
    current_points = points
    current_coverage = calculate_coverage(current_points)
    best_points = current_points
    best_coverage = current_coverage
    
    temperature = initial_temperature
    
    for i in range(iterations):
        # 生成新解
        new_points = current_points.copy()
        index = random.randint(0, len(new_points) - 1)
        new_points[index] = generate_random_points(1, domain_polygon)[0]  # 生成一个新点
        
        new_coverage = calculate_coverage(new_points)
        
        # 接受新解（只有当新解更好时才接受）
        if new_coverage > current_coverage:
            current_points = new_points
            current_coverage = new_coverage
        
        # 更新最优解
        if current_coverage > best_coverage:
            best_points = current_points
            best_coverage = current_coverage
        
        # 降温
        temperature *= cooling_rate
    
    return best_points, best_coverage

# 动态调整点数以满足目标覆盖率
def optimize_coverage():
    # 初始点数
    num_points = calculate_initial_points(raster_area, view_distance, target_coverage)
    print(f"初始点数量: {num_points}")
    
    while True:
        # 生成初始随机点
        points = generate_random_points(num_points, domain_polygon)
        
        # 运行模拟退火算法
        best_points, best_coverage = simulated_annealing(points)
        
        # 检查是否达到目标覆盖率
        if best_coverage >= target_coverage:
            print(f"达到目标覆盖率: {best_coverage * 100}%")
            return best_points, best_coverage
        else:
            print(f"当前覆盖率 {best_coverage * 100}% 未达到目标，增加点数...")
            num_points += 1  # 增加点数

# 主程序
if __name__ == "__main__":
    # 获取栅格的有效面积
    raster_area = calculate_raster_area(input_raster)
    print(f"栅格的有效面积: {raster_area} 平方米")
    
    # 获取栅格的有效区域多边形
    domain_polygon = get_raster_domain(input_raster)
    
    # 运行动态调整点数的优化算法
    best_points, best_coverage = optimize_coverage()
    
    # 输出结果
    print(f"最优点的数量: {len(best_points)}")
    print(f"最优点的位置: {best_points}")
    print(f"可见区域覆盖率: {best_coverage * 100}%")
    
    # 删除旧的 output_points.shp 文件
    if arcpy.Exists(output_points):
        arcpy.management.Delete(output_points)
    
    # 保存最优点的位置到点要素类
    spatial_ref = arcpy.Describe(input_raster).spatialReference
    arcpy.CreateFeatureclass_management(
        arcpy.env.workspace,
        output_points,
        "POINT",
        spatial_reference=spatial_ref  # 设置坐标系
    )
    
    # 添加 X 和 Y 坐标字段
    arcpy.management.AddField(output_points, "X", "DOUBLE")
    arcpy.management.AddField(output_points, "Y", "DOUBLE")
    
    # 插入点数据
    with arcpy.da.InsertCursor(output_points, ["SHAPE@XY", "X", "Y"]) as cursor:
        for point in best_points:
            if not math.isnan(point[0]) and not math.isnan(point[1]):  # 检查坐标是否有效
                cursor.insertRow([point, point[0], point[1]])
    
    # 将生成的要素类加载到地图中
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    map = aprx.listMaps()[0]  # 获取第一个地图
    map.addDataFromPath(os.path.join(arcpy.env.workspace, output_points))
    
    print("处理完成！")

栅格的有效面积: 572800.0 平方米
初始点数量: 29
当前覆盖率为: 0.9310405027932961
当前覆盖率为: 0.9310405027932961
当前覆盖率为: 0.9294692737430168
达到目标覆盖率: 93.10405027932961%
最优点的数量: 29
最优点的位置: [(333610.23234651936, 3804128.051669545), (333428.75307337753, 3803960.1452833754), (333517.5468628283, 3804381.6747308145), (333732.0745573518, 3804305.0029626624), (333369.3593935666, 3804305.716339669), (333365.61780601775, 3804119.4386900505), (333519.0262491349, 3804679.7586426567), (333382.21166048135, 3804490.6450462677), (333148.5439723854, 3804328.855376213), (333208.3306760147, 3803865.399363117), (333149.62894541444, 3804134.9112962964), (333039.57909298234, 3804467.213561186), (333050.37980891025, 3804629.1136324145), (333048.7007083091, 3803909.790901231), (333200.4027321171, 3804551.864732016), (333426.78780905047, 3804211.4384529106), (333634.1788472371, 3804383.6396225994), (333393.7169219339, 3804410.737226277), (333277.6349914583, 3803959.7889817627), (333221.1985406058, 3804127.794394554), (333163.6444476911, 